In [1]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import re

model = SentenceTransformer("intfloat/multilingual-e5-base")

traits = {
    "Yêu nước": [
        "yêu thiên nhiên",
        "có hành động bảo vệ thiên nhiên",
        "yêu quê hương",
        "yêu tổ quốc",
        "tôn trọng biểu trưng của đất nước",
        "kính trọng người lao động",
        "biết ơn người có công với đất nước",
        "tham gia hoạt động đền ơn đáp nghĩa"
    ],
    "Nhân ái": [
        "yêu thương người thân trong gia đình",
        "quan tâm chăm sóc người thân",
        "yêu quý bạn bè",
        "yêu quý thầy cô",
        "quan tâm động viên bạn bè",
        "khích lệ bạn bè",
        "tôn trọng người lớn tuổi",
        "giúp đỡ người già",
        "giúp đỡ người ốm yếu",
        "giúp đỡ người khuyết tật",
        "nhường nhịn em nhỏ",
        "giúp đỡ em nhỏ",
        "chia sẻ với bạn có hoàn cảnh khó khăn",
        "chia sẻ với người vùng sâu vùng xa",
        "chia sẻ với người bị ảnh hưởng thiên tai",
        "tôn trọng sự khác biệt của bạn bè",
        "không phân biệt đối xử",
        "không chia rẽ bạn bè",
        "sẵn sàng tha thứ cho bạn",
        "đoàn kết với bạn bè"
    ],
    "Chăm chỉ": [
        "đi học đầy đủ",
        "đi học đúng giờ",
        "hoàn thành nhiệm vụ học tập",
        "ham học hỏi",
        "thích đọc sách",
        "mở rộng hiểu biết",
        "vận dụng kiến thức vào đời sống",
        "tham gia công việc gia đình",
        "tham gia công việc trường lớp",
        "tham gia hoạt động cộng đồng"
    ],
    "Trung thực": [
        "thật thà trong học tập",
        "thật thà trong sinh hoạt",
        "ngay thẳng",
        "dám nói lên ý kiến",
        "giữ lời hứa",
        "nhận lỗi khi sai",
        "sửa lỗi",
        "bảo vệ điều đúng",
        "không lấy đồ của người khác",
        "không gian dối",
        "phản đối hành vi thiếu trung thực",
        "có tính trung thực"
    ],
    "Trách nhiệm": [
        "giữ gìn vệ sinh cá nhân",
        "rèn luyện thân thể",
        "chăm sóc sức khỏe",
        "sinh hoạt có nền nếp",
        "giữ gìn đồ dùng cá nhân",
        "giữ gìn tài sản gia đình",
        "không lãng phí thức ăn",
        "tiết kiệm tiền bạc",
        "tiết kiệm điện nước",
        "tuân thủ nội quy nhà trường",
        "tuân thủ quy định tập thể",
        "giữ vệ sinh chung",
        "bảo vệ của công",
        "không gây mất trật tự",
        "không cãi nhau",
        "không đánh nhau",
        "nhắc nhở bạn bè chấp hành nội quy",
        "nhắc nhở người thân chấp hành quy định",
        "hoàn thành công việc được giao",
        "có trách nhiệm với nhiệm vụ",
        "tham gia hoạt động tập thể",
        "tham gia hoạt động xã hội",
        "bảo vệ cây xanh",
        "bảo vệ động vật có ích",
        "giữ vệ sinh môi trường",
        "không xả rác",
        "phản đối hành vi phá hoại thiên nhiên",
        "chấp hành tốt nội quy lớp học"
    ]
}

keywords = {
    "Yêu nước": [
        "tổ quốc", "quê hương", "đất nước", "thiên nhiên", "đền ơn", "đáp nghĩa"
    ],
    "Nhân ái": [
        "đoàn kết", "yêu mến bạn bè", "yêu quý bạn bè", "quan tâm", "giúp đỡ",
        "chia sẻ", "khích lệ", "tha thứ", "nhường nhịn"
    ],
    "Chăm chỉ": [
        "đi học đầy đủ", "đúng giờ", "ham học hỏi", "chăm chỉ", "siêng năng",
        "hoàn thành nhiệm vụ học tập", "thích đọc sách"
    ],
    "Trung thực": [
        "trung thực", "thật thà", "ngay thẳng", "giữ lời hứa", "nhận lỗi",
        "sửa lỗi", "không gian dối"
    ],
    "Trách nhiệm": [
        "chấp hành tốt nội qui", "chấp hành tốt nội quy", "nội qui lớp học",
        "nội quy lớp học", "tuân thủ nội quy", "có trách nhiệm",
        "hoàn thành công việc được giao", "giữ vệ sinh chung"
    ]
}

trait_embs = {
    trait: model.encode(
        [f"passage: {x}" for x in examples],
        normalize_embeddings=True
    )
    for trait, examples in traits.items()
}

def split_clauses(text: str):
    text = text.lower().strip()
    parts = re.split(r"[.;,\n]", text)
    return [p.strip() for p in parts if p.strip()]

def score_clause(clause, emb_list, top_k=1):
    q = model.encode(f"query: {clause}", normalize_embeddings=True)
    sims = util.cos_sim(q, emb_list)[0].cpu().numpy()
    top_scores = np.sort(sims)[-top_k:]
    return float(np.mean(top_scores))

def keyword_score(text, trait_keywords):
    text = text.lower()
    hit = 0
    for kw in trait_keywords:
        if kw in text:
            hit += 1
    return min(hit / 2.0, 1.0)

def classify_hybrid(text):
    clauses = split_clauses(text)
    final_scores = {}

    for trait, emb_list in trait_embs.items():
        clause_scores = [score_clause(clause, emb_list, top_k=1) for clause in clauses]
        semantic = max(clause_scores) if clause_scores else 0.0
        lexical = keyword_score(text, keywords.get(trait, []))

        semantic_norm = max(0.0, min((semantic - 0.75) / 0.15, 1.0))
        final = 0.7 * semantic_norm + 0.3 * lexical

        final_scores[trait] = {
            "score": round(final, 4),
            "semantic": round(semantic, 4),
            "keyword": round(lexical, 4),
            "best_clause": clauses[int(np.argmax(clause_scores))] if clause_scores else None
        }

    return dict(sorted(final_scores.items(), key=lambda x: x[1]["score"], reverse=True))

d:\Work\Iu\hoc-ba-so\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3627.91it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text = "Em luôn đoàn kết và yêu mến bạn bè. Em có tính trung thực và chấp hành tốt nội qui lớp học."
print(classify_hybrid(text))

{'Trách nhiệm': {'score': 1.0, 'semantic': 0.9042, 'keyword': 1.0, 'best_clause': 'em có tính trung thực và chấp hành tốt nội qui lớp học'}, 'Nhân ái': {'score': 0.9909, 'semantic': 0.8981, 'keyword': 1.0, 'best_clause': 'em luôn đoàn kết và yêu mến bạn bè'}, 'Trung thực': {'score': 0.7966, 'semantic': 0.8886, 'keyword': 0.5, 'best_clause': 'em có tính trung thực và chấp hành tốt nội qui lớp học'}, 'Chăm chỉ': {'score': 0.5119, 'semantic': 0.8597, 'keyword': 0.0, 'best_clause': 'em có tính trung thực và chấp hành tốt nội qui lớp học'}, 'Yêu nước': {'score': 0.4804, 'semantic': 0.853, 'keyword': 0.0, 'best_clause': 'em luôn đoàn kết và yêu mến bạn bè'}}


: 